In [ ]:
# Notebook 06: Preference Data Generation

!pip install transformers datasets torch -q

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

import json
import random
from pathlib import Path
from typing import List, Dict
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

OUTPUT_DIR = BASE_DIR / "data/rlhf/preference_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preference data generation setup")

Preference data generation setup


In [ ]:

# Load chunks for context
with open(BASE_DIR / 'data/processed/chunks/all_chunks.json', 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

# Sample contexts
contexts = random.sample(chunks, min(1000, len(chunks)))
print(f"Sampled {len(contexts)} contexts for preference generation")

Loaded 12728 chunks
Sampled 1000 contexts for preference generation


In [ ]:

# Generate questions from contexts
def generate_question_from_context(context_text):
    """Generate research questions from paper chunks"""
    # Simple template-based generation
    templates = [
        f"What does this paper say about {context_text.split()[0]}?",
        "Can you explain the main concept discussed here?",
        "What are the key findings in this research?",
        "How does this work compare to prior research?",
        "What methodology is used in this study?"
    ]
    return random.choice(templates)

# Generate question-context pairs
qa_pairs = []
for context in tqdm(contexts[:500], desc="Generating QA pairs"):
    question = generate_question_from_context(context['text'])
    qa_pairs.append({
        'question': question,
        'context': context['text'][:500],  # Limit context length
        'arxiv_id': context['arxiv_id'],
        'title': context['title']
    })

print(f"Generated {len(qa_pairs)} QA pairs")

Generating QA pairs: 100%|██████████| 500/500 [00:00<00:00, 13025.14it/s]

Generated 500 QA pairs


In [ ]:

# Create preference pairs (good vs bad responses)
preference_data = []

for qa in tqdm(qa_pairs, desc="Creating preference pairs"):
    # Good response: accurate, well-cited, helpful
    good_response = f"Based on the research paper '{qa['title']}', {qa['context'][:200]}... This demonstrates the key findings from the study."

    # Bad response: vague, no citations, potentially inaccurate
    bad_responses = [
        "I'm not sure about that.",
        "This is a complex topic that requires more research.",
        "The paper discusses some interesting ideas but I can't recall specifics.",
    ]
    bad_response = random.choice(bad_responses)

    preference_data.append({
        'prompt': qa['question'],
        'chosen': good_response,
        'rejected': bad_response,
        'context': qa['context'],
        'source': qa['arxiv_id']
    })

print(f"Created {len(preference_data)} preference pairs")

Creating preference pairs: 100%|██████████| 500/500 [00:00<00:00, 170112.91it/s]

Created 500 preference pairs


In [ ]:

# Save preference data
train_split = int(0.8 * len(preference_data))
train_data = preference_data[:train_split]
val_data = preference_data[train_split:]

with open(OUTPUT_DIR / 'train_preferences.json', 'w') as f:
    json.dump(train_data, f, indent=2)

with open(OUTPUT_DIR / 'val_preferences.json', 'w') as f:
    json.dump(val_data, f, indent=2)

print(f"\nSaved preference data:")
print(f"  Train: {len(train_data)} pairs")
print(f"  Val: {len(val_data)} pairs")
print("\nNotebook 06 Complete!")


Saved preference data:
  Train: 400 pairs
  Val: 100 pairs

Notebook 06 Complete!
